# coach-app · Entrenar el detector de máquinas con YOLOv8 en Colab

Equivale a los pasos del video de YOLOv5, con las buenas prácticas de YOLOv8 que ya usa `ml/detection/pipeline.py`:

| Paso del video | Dónde está aquí |
|---|---|
| 1. Dataset `data/` con images y labels en train y val | Celdas 3–4 (carpeta creada con `pipeline.py yolo-folder`; sin fuga entre particiones) |
| 2. Elegir el modelo (en el video, el más grande) | Celda 5: `yolov8n.pt` (liviano para el celular); cambia a `yolov8s.pt` solo si mides la latencia |
| 3. Archivo `.yaml` con clases y rutas | `data.yaml` (viene dentro del zip; **no** lleva `path:`) |
| 4. Entrenar: imagen, batch y épocas | Celda 6 (misma configuración que `train_config.json`) |
| 5. TensorBoard y métricas | Celda 7 |
| 6. `best.pt` | Celda 9 (se descarga junto con las curvas) |
| 7. Inferencia con cajas, clase y confianza | Celda 8 |

**Antes de empezar**, en tu PC: `python ml/detection/pipeline.py yolo-folder` y comprime `ml/detection/dataset/yolo-v8/` en `yolo-v8.zip`.
En Colab: Entorno de ejecución → Cambiar tipo → GPU (T4).

In [ ]:
# 1) Entorno: GPU y versiones fijas (las mismas de ml/requirements.lock.txt)
import torch
print("GPU disponible:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!pip install -q "ultralytics==8.4.190"
import ultralytics; ultralytics.checks()

In [ ]:
# 2) Subir yolo-v8.zip (o monta Drive y cambia ZIP) y descomprimir
from google.colab import files
import os, zipfile
ZIP = "/content/yolo-v8.zip"
if not os.path.exists(ZIP):
    uploaded = files.upload()          # elige yolo-v8.zip
    ZIP = "/content/" + next(iter(uploaded))
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
DATA = "/content/yolo-v8"
print(open(f"{DATA}/data.yaml").read())

In [ ]:
# 3) Comprobar el dataset ANTES de gastar GPU: etiquetas válidas, sin duplicados entre particiones
import hashlib, glob, pathlib
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
seen, problems = {}, []
for split in ("train", "val", "test"):
    imgs = sorted(glob.glob(f"{DATA}/images/{split}/*"))
    print(split, len(imgs), "imágenes")
    for img in imgs:
        label = f"{DATA}/labels/{split}/{pathlib.Path(img).stem}.txt"
        if not os.path.exists(label): problems.append(f"sin etiqueta: {img}"); continue
        for line in open(label).read().splitlines():
            v = line.split()
            if len(v) != 5 or not all(0 <= float(x) <= 1 for x in v[1:]): problems.append(f"caja inválida en {label}: {line}")
        d = sha(img)
        if d in seen and seen[d] != split: problems.append(f"duplicada entre {seen[d]} y {split}: {img}")
        seen[d] = split
assert "path:" not in open(f"{DATA}/data.yaml").read().replace("# Sin `path:`", ""), "quita 'path:' del data.yaml"
print("PROBLEMAS:", problems[:10] if problems else "ninguno")
assert not problems

In [ ]:
# 4) Reproducibilidad: registrar versiones y huella del dataset
import json, platform, importlib.metadata as md, subprocess
manifest = json.load(open(f"{DATA}/dataset-manifest.json"))
print("Clases:", manifest["names"], "| conteo:", manifest["conteo"], "| semilla del split:", manifest["semilla"])
for w in manifest["advertencias"]: print("AVISO:", w)

In [ ]:
# 5) Configuración (idéntica a ml/detection/train_config.json). SMOKE=True hace una prueba corta de 5 épocas.
from ultralytics import YOLO, settings
settings.update({"tensorboard": True})        # TensorBoard registra métricas en /content/runs
SMOKE = False
MODEL = "yolov8n.pt"                          # n/s/m/l/x; el video usa la más grande, aquí priorizamos el móvil
cfg = dict(imgsz=640, batch=16 if torch.cuda.is_available() else 8, epochs=5 if SMOKE else 150,
           patience=30, seed=42, deterministic=True, optimizer="AdamW", lr0=0.001, lrf=0.01,
           weight_decay=0.0005, warmup_epochs=3.0, cos_lr=True, amp=True,
           hsv_h=0.015, hsv_s=0.3, hsv_v=0.3, degrees=5.0, translate=0.1, scale=0.3, shear=0.0,
           perspective=0.0, fliplr=0.5, flipud=0.0, mosaic=0.5, mixup=0.0,
           close_mosaic=0 if SMOKE else 10, workers=2, cache=False, val=True, plots=True, save=True, save_period=10)
print(cfg)

In [ ]:
# 6) Entrenar (el video usa 50–60 épocas para el ejemplo; aquí 150 con parada temprana si deja de mejorar)
RUN = "v8n-colab"
model = YOLO(MODEL)
assert model.task == "detect"
model.train(data=f"{DATA}/data.yaml", project="/content/runs", name=RUN, exist_ok=False, device=0 if torch.cuda.is_available() else "cpu", **cfg)
BEST = f"/content/runs/{RUN}/weights/best.pt"

In [ ]:
# 7) Métricas: curvas de aprendizaje y TensorBoard. Si train baja y val sube, hay sobreajuste: revisa los datos antes de añadir épocas.
%load_ext tensorboard
%tensorboard --logdir /content/runs
import pandas as pd
df = pd.read_csv(f"/content/runs/{RUN}/results.csv"); df.columns = [c.strip() for c in df.columns]
print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5))
best = YOLO(BEST)
val_metrics = best.val(data=f"{DATA}/data.yaml", split="val", imgsz=cfg["imgsz"], batch=cfg["batch"])   # val sirve para ELEGIR; test solo al final
print("mAP50-95 (val):", val_metrics.box.map)

In [ ]:
# 8) Inferencia como en el video: caja, clase y confianza por objeto (imágenes de val)
import numpy as np
results = best.predict(source=f"{DATA}/images/val", conf=0.25, save=True, project="/content/runs", name=f"{RUN}-pred", exist_ok=True, stream=False)
rows = []
for r in results[:10]:
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        rows.append({"imagen": os.path.basename(r.path), "clase": r.names[int(b.cls)], "confianza": round(float(b.conf), 3),
                     "xmin": round(x1), "ymin": round(y1), "xmax": round(x2), "ymax": round(y2)})
display(pd.DataFrame(rows))

In [ ]:
# 9) Resultado final con test (ejecuta UNA sola vez, con la configuración ya elegida) y guardar pesos + evidencias
test_metrics = best.val(data=f"{DATA}/data.yaml", split="test", imgsz=cfg["imgsz"], batch=cfg["batch"])
summary = {"modelo_inicial": MODEL, "corrida": RUN, "python": platform.python_version(),
           "ultralytics": md.version("ultralytics"), "torch": md.version("torch"),
           "configuracion": cfg, "dataset": {"conteo": manifest["conteo"], "names": manifest["names"], "semilla": manifest["semilla"]},
           "val": {"mAP50": float(val_metrics.box.map50), "mAP50-95": float(val_metrics.box.map)},
           "test": {"mAP50": float(test_metrics.box.map50), "mAP50-95": float(test_metrics.box.map)},
           "best_pt_sha256": sha(BEST)}
json.dump(summary, open(f"/content/runs/{RUN}/run-summary.json", "w"), indent=2, ensure_ascii=False)
!cd /content/runs && zip -qr {RUN}.zip {RUN}
files.download(f"/content/runs/{RUN}.zip")
print(summary)

## Después de entrenar

1. Descomprime `v8n-colab.zip` en `ml/detection/runs/` (la carpeta `runs/` no se versiona).
2. En tu PC: `python ml/detection/pipeline.py export --model ml/detection/runs/v8n-colab/weights/best.pt`
   → crea el `.tflite` y su `*.manifest.json` (sha256, contrato de entrada/salida, clases y métricas de val).
3. Mide la latencia en el teléfono antes de decidir entre `yolov8n` y `yolov8s`.
4. La app **todavía no** consume detecciones: `machineRecognition.ts` espera el clasificador `[1,224,224,3] → [1,35]`.
   Integrar el detector requiere decodificar la salida sin NMS (ver el manifiesto) y es una etapa aparte.